# PDB Tools

![PDB Tools](https://proto-bio.github.io/proto-assets/images/tool/pdb/hero.png)

This notebook covers the four PDB tools provided by proto_tools. `run_pdb_fetch_entry` retrieves entry-level metadata from the RCSB REST API, returning the structure title, experimental method, and resolution for a given accession. `run_pdb_fetch_fasta` fetches the FASTA sequences for all polymer chains in a structure and automatically classifies each chain as protein or nucleotide based on its composition. `run_pdb_sequence_search` finds polymer entities whose sequences resemble a query, using RCSB's MMseqs2 sequence service ([Steinegger and Söding, 2017](https://doi.org/10.1038/nbt.3988)). `run_pdb_structure_similarity` searches experimental PDB chains or assemblies for structures similar to a query, using RCSB's learned structure embedding ([Segura et al., 2026](https://doi.org/10.1093/bioinformatics/btag058)). Together these functions support common protein design workflows where you need to retrieve a structural template, its sequences, and its structural neighbors before running inverse folding, mutagenesis, or structure prediction.

In [1]:
from proto_tools.utils.notebook_docs import display_overview, display_api_reference, display_docs_section, display_doc_link, display_available_tools
display_doc_link("pdb")
display_overview("pdb")
display_docs_section("pdb", "Background")

# PDB

The RCSB Protein Data Bank is a database of experimentally determined three-dimensional structures of proteins, nucleic acids, and their complexes. This toolkit provides four CPU-only tools: `pdb-fetch-entry` retrieves structure metadata (title, experimental method, and resolution), `pdb-fetch-fasta` retrieves chain sequences, `pdb-sequence-search` finds polymer entities whose sequences resemble a query, and `pdb-structure-similarity` searches experimental PDB chains or assemblies with RCSB's learned structure embedding. They require network access.

The Protein Data Bank ([Berman et al., 2000](https://doi.org/10.1093/nar/28.1.235)) is the single worldwide archive of experimentally determined macromolecular structures, served here through the RCSB PDB. It is operated by the Research Collaboratory for Structural Bioinformatics (RCSB) at Rutgers University and the University of California San Diego, with funding from the National Science Foundation, the National Institutes of Health, and the Department of Energy. Entries are solved by X-ray crystallography, cryo-electron microscopy, nuclear magnetic resonance spectroscopy, and other experimental methods.

The tools call two RCSB HTTP endpoints directly. `pdb-fetch-entry` issues a GET request to the RCSB Data API core entry endpoint (`https://data.rcsb.org/rest/v1/core/entry/{pdb_id}`) and reads the structure title from `struct.title`, the experimental method from the first `exptl` record, and the resolution from `rcsb_entry_info.resolution_combined`, which covers both X-ray and cryo-EM entries; entries solved by NMR have no resolution value. `pdb-fetch-fasta` requests the FASTA endpoint (`https://www.rcsb.org/fasta/entry/{pdb_id}`), parses each record, extracts the author-assigned chain identifiers from the header, and classifies a sequence as protein when it contains amino-acid letters that do not also occur in nucleotide alphabets. Both tools uppercase the supplied accession, retry transient HTTP failures with backoff, and return an empty result when the accession is not found (HTTP 404). Results reflect the live archive at query time rather than a fixed release snapshot.

`pdb-structure-similarity` calls the RCSB Search API's structure service. The service represents structures with ESM3 residue embeddings followed by a learned aggregator whose cosine similarity was trained to approximate maximum TM-score ([Segura et al., 2026](https://doi.org/10.1093/bioinformatics/btag058)). It is an independent approximate-nearest-neighbor search, not Foldseek or an exact structural alignment. Public PDB references are queried by entry and chain/assembly ID. Custom structures are converted to PDB format and sent to RCSB's public upload service.

`pdb-sequence-search` calls the same Search API's sequence service, which aligns the query against every polymer entity in the archive with MMseqs2 ([Steinegger and Söding, 2017](https://doi.org/10.1038/nbt.3988)) and returns ranked entities with their alignment statistics, typically in under a second. Hits are then annotated with one request to the RCSB Data API GraphQL endpoint (`https://data.rcsb.org/graphql`), which supplies each entity's description, source organism, UniProt accessions, author chain identifiers, and canonical sequence.

## Available tools

In [2]:
display_available_tools("pdb")

- **`run_pdb_fetch_entry()`** — Fetch structure metadata (title, method, resolution) from RCSB PDB
- **`run_pdb_fetch_fasta()`** — Fetch chain sequences from RCSB PDB with protein/nucleotide classification
- **`run_pdb_sequence_search()`** — Fast sequence similarity search (seconds): identify which protein a sequence or fragment comes from, searching PDB polymer entities with RCSB's MMseqs2 service
- **`run_pdb_structure_similarity()`** — Search experimental PDB chains or assemblies for structures similar to a query, using RCSB's learned structure embedding

### `run_pdb_fetch_entry`

`run_pdb_fetch_entry` queries the RCSB REST API to retrieve catalog-level information about a PDB structure: the structure title, experimental method (e.g., X-RAY DIFFRACTION, SOLUTION NMR), and resolution in angstroms. This is useful as a quick lookup step before downloading full structure files, allowing you to confirm the experimental quality and method of a candidate template. We use PDB entry 1EFA, the lac repressor bound to a symmetric operator DNA, as the running example.

In [3]:
from proto_tools import (
    PdbFetchEntryConfig,
    PdbFetchEntryInput,
    run_pdb_fetch_entry,
)

In [4]:
# Display docs
display_api_reference("pdb", "input", "run_pdb_fetch_entry")

# Example input
inputs = PdbFetchEntryInput(pdb_id="1EFA")

**Input** — `PdbFetchEntryInput`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| <code>pdb_id</code> | <code>str</code> | required | PDB accession (e.g. '1LBG') |

In [5]:
# Display config docs
display_api_reference("pdb", "config", "run_pdb_fetch_entry")

# Default config | see docs above for all fields
config = PdbFetchEntryConfig()

*No config fields.*

In [6]:
# Run the entry fetch
entry_output = run_pdb_fetch_entry(inputs, config)

In [7]:
# Display output docs
display_api_reference("pdb", "output", "run_pdb_fetch_entry")

# Inspect the structure metadata
print(f"Title: {entry_output.title}")
print(f"Method: {entry_output.method}")
print(f"Resolution: {entry_output.resolution} A")
print(f"Source: {entry_output.source_url}")

**Output** — `PdbFetchEntryOutput`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| <code>title</code> | <code>str &#124; None</code> | <code>None</code> | Human-readable RCSB entry title |
| <code>method</code> | <code>str &#124; None</code> | <code>None</code> | PDB experimental method (e.g. X-RAY DIFFRACTION, SOLUTION NMR, ELECTRON MICROSCOPY) |
| <code>resolution</code> | <code>float &#124; None</code> | <code>None</code> | Resolution in Å (None for NMR or fiber diffraction) |
| <code>source_url</code> | <code>str</code> | required | Request URL |

Title: CRYSTAL STRUCTURE OF THE LAC REPRESSOR DIMER BOUND TO OPERATOR AND THE ANTI-INDUCER ONPF
Method: X-RAY DIFFRACTION
Resolution: 2.6 A
Source: https://data.rcsb.org/rest/v1/core/entry/1EFA


### `run_pdb_fetch_fasta`

`run_pdb_fetch_fasta` retrieves the polymer chain sequences from RCSB PDB in FASTA format and automatically classifies each chain as protein or nucleotide based on its amino acid composition. A PDB structure can contain multiple chains of different types, so this function is useful when you need to extract specific chains for downstream analysis, such as feeding a protein chain into a structure prediction model or aligning it against homologous sequences.

In [8]:
from proto_tools import (
    PdbFetchFastaConfig,
    PdbFetchFastaInput,
    run_pdb_fetch_fasta,
)

In [9]:
# Display docs
display_api_reference("pdb", "input", "run_pdb_fetch_fasta")

# Fetch chain sequences for the lac repressor structure
fasta_inputs = PdbFetchFastaInput(pdb_id="1EFA")

**Input** — `PdbFetchFastaInput`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| <code>pdb_id</code> | <code>str</code> | required | PDB accession (e.g. '1LBG') |

In [10]:
# Display config docs
display_api_reference("pdb", "config", "run_pdb_fetch_fasta")

# Default config | see docs above for all fields
fasta_config = PdbFetchFastaConfig()

*No config fields.*

In [11]:
# Run the FASTA fetch
fasta_output = run_pdb_fetch_fasta(fasta_inputs, fasta_config)

In [12]:
# Display output docs
display_api_reference("pdb", "output", "run_pdb_fetch_fasta")

# Inspect the chain sequences and their classifications
print(f"Chains: {len(fasta_output.chains)}")
print()

for chain in fasta_output.chains:
    chain_type = "protein" if chain.is_protein else "nucleotide"
    preview = chain.sequence[:50] + ("..." if len(chain.sequence) > 50 else "")
    print(f"  Chains {chain.chain_ids}: {chain_type}, length={len(chain.sequence)}")
    print(f"    {preview}")

**Output** — `PdbFetchFastaOutput`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| <code>chains</code> | <code>list[PdbChain]</code> | <code>[]</code> | Parsed chain sequences |
| <code>source_url</code> | <code>str</code> | required | Request URL |

**`PdbChain`**

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| <code>chain_ids</code> | <code>list[str]</code> | <code>[]</code> | Author-assigned chain IDs sharing this sequence |
| <code>header</code> | <code>str</code> | required | FASTA header |
| <code>sequence</code> | <code>str</code> | required | Chain sequence |
| <code>is_protein</code> | <code>bool</code> | required | True if protein, False if nucleic acid |

Chains: 2

  Chains ['D', 'E']: nucleotide, length=21
    GAATTGTGAGCGCTCACAATT
  Chains ['A', 'B', 'C']: protein, length=333
    MKPVTLYDVAEYAGVSYQTVSRVVNQASHVSAKTREKVEAAMAELNYIPN...


### `run_pdb_sequence_search`

`run_pdb_sequence_search` aligns each query sequence against every polymer entity in the PDB with RCSB's MMseqs2 sequence service and returns ranked entities with the identity, E-value, bit score, and aligned ranges of the best alignment. By default each hit is also annotated, in one extra request, with the entity description, source organism, UniProt accessions, author chain IDs, and canonical sequence. Because the search tolerates mismatches, it can identify the source of a fragment that a text search would miss. Here we search with a 30-residue fragment of human transthyretin that carries three substitutions.

In [13]:
from proto_tools import (
    PdbSequenceSearchConfig,
    PdbSequenceSearchInput,
    PdbSequenceSearchQuery,
    run_pdb_sequence_search,
)

In [14]:
# Display docs
display_api_reference("pdb", "input", "run_pdb_sequence_search")

# A transthyretin fragment with three substitutions (I26V, V32I, K48R)
sequence_inputs = PdbSequenceSearchInput(queries=[PdbSequenceSearchQuery(sequence="VRGSPAVNVAVHIFRKAADDTWEPFASGRT")])

**Input** — `PdbSequenceSearchInput`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| <code>queries</code> | <code>list[PdbSequenceSearchQuery]</code> | required | Query sequences to search; a single query or a bare sequence string is accepted |

In [15]:
# Display config docs
display_api_reference("pdb", "config", "run_pdb_sequence_search")

# Keep the 5 best entities at 50% identity or more over the aligned region
sequence_config = PdbSequenceSearchConfig(identity_cutoff=0.5, evalue_cutoff=1.0, max_hits=5)

**Config** — `PdbSequenceSearchConfig`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| <code>identity_cutoff</code> | <code>float</code> | <code>0.0</code> | Minimum sequence identity over the aligned region, from 0 to 1 |
| <code>evalue_cutoff</code> | <code>float</code> | <code>0.1</code> | Maximum E-value of a reported hit |
| <code>max_hits</code> | <code>int</code> | <code>25</code> | Maximum hits returned per query; RCSB caps one request at 10,000 |
| <code>annotate</code> | <code>bool</code> | <code>True</code> | Add description, organism, UniProt IDs, author chain IDs, and sequence with one extra request |
| <code>include_computed_models</code> | <code>bool</code> | <code>False</code> | Also search RCSB's computed structure models; RCSB's sequence server often times out on these |

In [16]:
# Run the sequence search
sequence_output = run_pdb_sequence_search(sequence_inputs, sequence_config)

In [17]:
# Display output docs
display_api_reference("pdb", "output", "run_pdb_sequence_search")

result = sequence_output.results[0]
print(f"{result.total_count} matching entities")
for hit in result.hits:
    print(
        f"  {hit.entity_id:8} identity {hit.sequence_identity:.2f}  E={hit.evalue:.1e}  "
        f"chains {','.join(hit.chain_ids or [])}  {hit.uniprot_ids}  {hit.description}"
    )

**Output** — `PdbSequenceSearchOutput`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| <code>results</code> | <code>list[PdbSequenceSearchResult]</code> | <code>[]</code> | One result per query, in input order |
| <code>source_url</code> | <code>str</code> | required | RCSB Search API endpoint used |

**`PdbSequenceSearchResult`**

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| <code>hits</code> | <code>list[PdbSequenceSearchHit]</code> | <code>[]</code> | Ranked polymer-entity matches |
| <code>total_count</code> | <code>int</code> | <code>0</code> | Number of matches reported by RCSB, including any beyond the returned hits |

**`PdbSequenceSearchHit`**

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| <code>entity_id</code> | <code>str</code> | required | RCSB polymer entity ID, e.g. 1BM7_1 |
| <code>pdb_id</code> | <code>str</code> | required | PDB entry ID of the entity |
| <code>score</code> | <code>float &#124; None</code> | <code>None</code> | RCSB normalized ranking score |
| <code>sequence_identity</code> | <code>float &#124; None</code> | <code>None</code> | Identity over the aligned region |
| <code>evalue</code> | <code>float &#124; None</code> | <code>None</code> | Alignment E-value |
| <code>bitscore</code> | <code>float &#124; None</code> | <code>None</code> | Alignment bit score |
| <code>alignment_length</code> | <code>int &#124; None</code> | <code>None</code> | Aligned columns |
| <code>mismatches</code> | <code>int &#124; None</code> | <code>None</code> | Mismatched aligned positions |
| <code>gaps_opened</code> | <code>int &#124; None</code> | <code>None</code> | Gap openings in the alignment |
| <code>query_beg</code> | <code>int &#124; None</code> | <code>None</code> | First aligned query residue (1-indexed) |
| <code>query_end</code> | <code>int &#124; None</code> | <code>None</code> | Last aligned query residue (inclusive) |
| <code>subject_beg</code> | <code>int &#124; None</code> | <code>None</code> | First aligned entity residue (1-indexed) |
| <code>subject_end</code> | <code>int &#124; None</code> | <code>None</code> | Last aligned entity residue (inclusive) |
| <code>query_aligned_seq</code> | <code>str &#124; None</code> | <code>None</code> | Query side of the alignment |
| <code>subject_aligned_seq</code> | <code>str &#124; None</code> | <code>None</code> | Entity side of the alignment |
| <code>description</code> | <code>str &#124; None</code> | <code>None</code> | Entity description (annotated) |
| <code>organism</code> | <code>str &#124; None</code> | <code>None</code> | Source organism (annotated) |
| <code>uniprot_ids</code> | <code>list[str] &#124; None</code> | <code>None</code> | UniProt accessions (annotated) |
| <code>chain_ids</code> | <code>list[str] &#124; None</code> | <code>None</code> | Author chain IDs of the entity, as pdb-fetch-fasta reports them (annotated) |
| <code>sequence</code> | <code>str &#124; None</code> | <code>None</code> | Canonical entity sequence (annotated) |

503 matching entities
  1BM7_1   identity 0.90  E=9.4e-14  chains A,B  ['P02766']  PROTEIN (TRANSTHYRETIN)
  1BMZ_1   identity 0.90  E=9.4e-14  chains A,B  ['P02766']  PROTEIN (TRANSTHYRETIN)
  1BZ8_1   identity 0.90  E=9.4e-14  chains A,B  ['P02766']  PROTEIN (TRANSTHYRETIN)
  1BZD_1   identity 0.90  E=9.4e-14  chains A,B  ['P02766']  PROTEIN (TRANSTHYRETIN)
  1BZE_1   identity 0.90  E=9.4e-14  chains A,B  ['P02766']  PROTEIN (TRANSTHYRETIN)


### `run_pdb_structure_similarity`

`run_pdb_structure_similarity` searches the experimental PDB for chains or assemblies structurally similar to each query. A query is either a public PDB chain or assembly, or a custom `Structure`. Custom structures are uploaded to RCSB's public upload service before the search. Each hit carries RCSB's normalized ranking score and a `predicted_tm_score` from the learned embedding, which estimates TM-score without performing an alignment. Several queries can be passed in one call, and results come back in input order. Here we search with both chains of 1YCR: MDM2 (chain A) and the p53 transactivation-domain peptide bound to it (chain B).

In [18]:
from proto_tools import (
    PdbStructureSimilarityConfig,
    PdbStructureSimilarityInput,
    PdbStructureSimilarityQuery,
    run_pdb_structure_similarity,
)

In [19]:
# Display docs
display_api_reference("pdb", "input", "run_pdb_structure_similarity")

# Two public queries: MDM2 (chain A) and the bound p53 peptide (chain B)
similarity_inputs = PdbStructureSimilarityInput(
    queries=[
        PdbStructureSimilarityQuery(pdb_id="1YCR", chain_id="A"),
        PdbStructureSimilarityQuery(pdb_id="1YCR", chain_id="B"),
    ]
)

**Input** — `PdbStructureSimilarityInput`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| <code>queries</code> | <code>list[PdbStructureSimilarityQuery]</code> | required | Public PDB references or custom structures to search |

In [20]:
# Display config docs
display_api_reference("pdb", "config", "run_pdb_structure_similarity")

# Keep the 10 nearest chains with a predicted TM-score of at least 0.5
similarity_config = PdbStructureSimilarityConfig(number_of_candidates=10, predicted_tm_score_cutoff=0.5)

**Config** — `PdbStructureSimilarityConfig`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| <code>target_search_space</code> | <code>Literal['polymer_entity_instance', 'assembly']</code> | <code>'polymer_entity_instance'</code> | Search experimental PDB chains or biological assemblies |
| <code>similarity_type</code> | <code>Literal['global', 'local']</code> | <code>'global'</code> | global length-normalized similarity or local substructure similarity (any case) |
| <code>predicted_tm_score_cutoff</code> | <code>float</code> | <code>0.0</code> | Minimum RCSB embedding-model predicted TM score to return |
| <code>number_of_candidates</code> | <code>int</code> | <code>100</code> | Maximum nearest-neighbor candidates returned; RCSB caps one request at 10,000 |

In [21]:
# Run the structure-similarity search
similarity_output = run_pdb_structure_similarity(similarity_inputs, similarity_config)

In [22]:
# Display output docs
display_api_reference("pdb", "output", "run_pdb_structure_similarity")

# One result per query, in input order
for query, result in zip(similarity_inputs.queries, similarity_output.results):
    print(f"{query.pdb_id} chain {query.chain_id}: {result.total_count} matches")
    for hit in result.hits[:5]:
        print(f"  {hit.identifier:10} predicted TM-score {hit.predicted_tm_score:.3f}")
    print()

**Output** — `PdbStructureSimilarityOutput`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| <code>results</code> | <code>list[PdbStructureSimilarityResult]</code> | <code>[]</code> | One result per query, in input order |
| <code>source_url</code> | <code>str</code> | required | RCSB Search API endpoint used |

**`PdbStructureSimilarityResult`**

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| <code>hits</code> | <code>list[PdbStructureSimilarityHit]</code> | <code>[]</code> | Ranked experimental-PDB structure matches |
| <code>total_count</code> | <code>int</code> | <code>0</code> | Number of matches reported by RCSB, including any beyond the returned hits |

**`PdbStructureSimilarityHit`**

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| <code>identifier</code> | <code>str</code> | required | RCSB hit ID: entry.label_asym_id for chains, entry-assembly for assemblies |
| <code>score</code> | <code>float &#124; None</code> | <code>None</code> | RCSB normalized ranking score |
| <code>predicted_tm_score</code> | <code>float &#124; None</code> | <code>None</code> | Structure service original score, trained to approximate maximum TM-score |

1YCR chain A: 10 matches
  1YCR.A     predicted TM-score 1.000
  22IZ.C     predicted TM-score 0.944
  5SWK.B     predicted TM-score 0.944
  3LNZ.G     predicted TM-score 0.937
  5SWK.A     predicted TM-score 0.929

1YCR chain B: 10 matches
  1YCR.B     predicted TM-score 1.000
  2MKR.B     predicted TM-score 0.989
  3M93.B     predicted TM-score 0.989
  1GJG.A     predicted TM-score 0.919
  3DAC.B     predicted TM-score 0.915



## Export Results

Fetched results can be saved to JSON format for downstream use in other tools or analysis pipelines.

In [23]:
from pathlib import Path

output_dir = Path("./example_output")
output_dir.mkdir(exist_ok=True)

fasta_output.export("pdb_fasta_results", export_path=output_dir, file_format="json")
print(f"Exported to {output_dir / 'pdb_fasta_results.json'}")

Exported to example_output/pdb_fasta_results.json
